In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession
import wget

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += ";D:\\hadoop\\bin"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark.version

'3.5.3'

In [4]:
wget.download('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet')

100% [........................................................................] 71134255 / 71134255

'yellow_tripdata_2025-11.parquet'

In [11]:
df_yellow_nov = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [12]:
df_yellow_nov.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [13]:
type(df_yellow_nov)

pyspark.sql.dataframe.DataFrame

In [14]:
df_yellow_nov.repartition(4).write.parquet('hw/yellow/nov/')

In [16]:
df_yellow_nov.registerTempTable('yellow_nov')

In [19]:
df_yellow_count = spark.sql("""
SELECT 
    COUNT(1) AS number_records
FROM
    yellow_nov
WHERE
    tpep_pickup_datetime >= '2025-11-15 00:00:00'
AND
    tpep_pickup_datetime < '2025-11-16 00:00:00'
""")

df_yellow_count.show()

+--------------+
|number_records|
+--------------+
|        162604|
+--------------+



In [28]:
df_yellow_hours = spark.sql("""
SELECT 
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS diff_hour 
FROM
    yellow_nov
ORDER BY
    diff_hour DESC
LIMIT 1
""")

df_yellow_hours.show()

+-----------------+
|        diff_hour|
+-----------------+
|90.64666666666666|
+-----------------+



In [35]:
df_timezone_pd = pd.read_csv('taxi_zone_lookup.csv')
df_timezone_pd.dtypes

LocationID       int64
Borough         object
Zone            object
service_zone    object
dtype: object

In [36]:
spark.createDataFrame(df_timezone_pd).schema

StructType([StructField('LocationID', LongType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [37]:
from pyspark.sql import types
timezone_schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True), 
    types.StructField('Borough', types.StringType(), True), 
    types.StructField('Zone', types.StringType(), True), 
    types.StructField('service_zone', types.StringType(), True)
])


In [38]:
df_timezone = spark.read.option("header", "true").schema(timezone_schema).csv(('taxi_zone_lookup.csv'))
df_timezone.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [45]:
df_join = df_yellow_nov.join(df_timezone, df_yellow_nov.PULocationID == df_timezone.LocationID, how='inner')
df_join.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+-------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|               Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--

In [55]:
df_join.registerTempTable('join_temp')
df_least_pickup_zone = spark.sql("""
SELECT
    Zone,
    COUNT(*) AS number_records
FROM
    join_temp
GROUP BY
    Zone
ORDER BY
    number_records ASC
""")

df_least_pickup_zone.show()

+--------------------+--------------+
|                Zone|number_records|
+--------------------+--------------+
|Governor's Island...|             1|
|Eltingville/Annad...|             1|
|       Arden Heights|             1|
|       Port Richmond|             3|
|       Rikers Island|             4|
|   Rossville/Woodrow|             4|
| Green-Wood Cemetery|             4|
|         Great Kills|             4|
|         Jamaica Bay|             5|
|         Westerleigh|            12|
|        Crotona Park|            14|
|             Oakwood|            14|
|New Dorp/Midland ...|            14|
|       West Brighton|            14|
|       Willets Point|            15|
|Breezy Point/Fort...|            16|
|Saint George/New ...|            17|
|       Broad Channel|            18|
|     Mariners Harbor|            21|
|Heartland Village...|            22|
+--------------------+--------------+
only showing top 20 rows

